In [ ]:
import pandas as pd
import os

df = pd.read_parquet("data/dataset_limpio.parquet")
df

In [1]:
import requests, pandas as pd

r = requests.get("https://sinca.mma.gob.cl/index.php/json/listadomapa2k19/", timeout=60)
est = pd.json_normalize(r.json())
print(est.columns.tolist())   # para ver cómo se llaman los campos
est.head()

['nombre', 'key', 'latitud', 'longitud', 'comuna', 'red', 'region', 'regionindex', 'calificacion', 'empresa', 'realtime']


,nombre,key,latitud,longitud,comuna,red,region,regionindex,calificacion,empresa,realtime
0,Alto Hospicio,117,-20.290467,-70.100192,Alto Hospicio,Red MMA,Región de Tarapacá,2,Pública,Ministerio del Medio Ambiente,"[{'code': 'PM25', 'name': 'MP-2,5', 'info': {'..."
1,Gobernación,201,-22.093082,-70.201210,Tocopilla,Ciudad Tocopilla,Región de Antofagasta,3,Privada,Norgener S.A,"[{'code': 'PM10', 'name': 'MP-10', 'info': {'c..."
2,Jardín Infantil Integra,209,-23.098798,-70.442765,Mejillones,Red ENAEX,Región de Antofagasta,3,no definida,ENAEX S.A. Planta Prillex América,"[{'code': '0003', 'name': 'Di&oacute;xido de n..."
3,Hospital del Cobre,217,-22.454016,-68.910148,Calama,None,Región de Antofagasta,3,Privada,Codelco Distrito Norte,"[{'code': 'PM25', 'name': 'MP-2,5', 'info': {'..."
4,Bomberos,230,-22.111064,-70.210828,Tocopilla,Ciudad Tocopilla,Región de Antofagasta,3,Privada,Norgener S.A,"[{'code': 'PM25', 'name': 'MP-2,5', 'info': {'..."


In [2]:
import sys; sys.path.append("..")
from pyproj import Transformer
from src.config import ESTACIONES

est["key"] = est["key"].astype(str)
mias = est[est["key"].isin(ESTACIONES.keys())].copy()

# 1. ¿Están las 16? ¿Quién las opera?
print(f"Encontradas: {len(mias)} de {len(ESTACIONES)}")
faltan = set(ESTACIONES) - set(mias["key"])
if faltan:
    print("No encontradas:", {k: ESTACIONES[k][0] for k in faltan})
print(mias[["key", "nombre", "comuna", "red", "calificacion", "empresa", "latitud", "longitud"]]
      .sort_values("comuna").to_string(index=False))

# 2. Bloque corregido para config.py (UTM WGS84 huso 18S)
t = Transformer.from_crs("EPSG:4326", "EPSG:32718", always_xy=True)
print("\nUTM_ESTACIONES = {")
for _, r in mias.sort_values("key").iterrows():
    e, n = t.transform(float(r["longitud"]), float(r["latitud"]))
    print(f'    "{r["key"]}": ({e:.0f}, {n:.0f}),   # {r["nombre"]}, {r["comuna"]}')
print("}")

# 3. Estaciones de la zona que tu dataset NO tiene (ej. Tomé, Penco, San Pedro)
zona = ["Concepción", "Talcahuano", "Hualpén", "Chiguayante", "Hualqui", "Coronel",
        "Lota", "Tomé", "Penco", "San Pedro de la Paz"]
otras = est[est["comuna"].isin(zona) & ~est["key"].isin(ESTACIONES.keys())]
print("\nOtras estaciones de la zona saturada:")
print(otras[["key", "nombre", "comuna", "calificacion", "empresa"]].to_string(index=False))

Encontradas: 16 de 16
key                    nombre      comuna              red calificacion                           empresa    latitud   longitud
854                  Punteras Chiguayante       MMA Biobío      Pública Sub Secretaría del Medio Ambiente -36.923330 -73.036132
827          Kingston College  Concepción       MMA Biobío      Pública Sub Secretaría del Medio Ambiente -36.784654 -73.052063
831             Cerro Merquín     Coronel       MMA Biobío      Pública Sub Secretaría del Medio Ambiente -37.021183 -73.149556
859                Lota rural     Coronel     ENEL Coronel      Privada        Enel Generacion Chile S.A. -37.100696 -73.155760
860               Lota urbana     Coronel     ENEL Coronel      Privada        Enel Generacion Chile S.A. -37.077620 -73.154136
878             Coronel Norte     Coronel Colbun - Coronel      Privada                        Colbún S.A -37.009021 -73.149097
879          Lagunillas, ENEL     Coronel     ENEL Coronel      Privada        Ene